## Data ignestion to Gold layer

### Importing Dependencies

In [0]:
import pyspark.sql.functions as sf

### Reading silver data

In [0]:
df = spark.read.table("aml_transactions.aml_transactions_schema.silver_aml_transactions")

### Building "gold_aml_transactions" table

In [0]:
df = (df
    .withColumn("transaction_date", sf.to_date("timestamp"))
    .groupBy("transaction_date", "payment_currency")
    .agg(
        sf.count("*").alias("tot_transaction_count"),
        sf.round(sf.sum("amount_paid"), 2).alias("total_amount_paid"),
        sf.count(sf.when(sf.col("is_laundering") == 1, 1)).alias("laundering_count"),
        sf.count(sf.when(sf.col("is_laundering") == 0, 1)).alias("non_laundering_count"),
        sf.round(sf.sum(sf.when(sf.col("is_laundering") == 1, sf.col("amount_paid"))), 2).alias("net_laundering_amount"),
        sf.round((sf.col("laundering_count")/sf.col("tot_transaction_count"))*100, 2).alias("laundering_labelled_transaction_pct")
        )
    .sort(sf.col("transaction_date").desc())
)

display(df)

transaction_date,payment_currency,tot_transaction_count,total_amount_paid,laundering_count,non_laundering_count,net_laundering_amount,laundering_labelled_transaction_pct
2022-09-18,Saudi Riyal,1,2391.92,1,0,2391.92,100.0
2022-09-18,US Dollar,8,45071.27,5,3,27839.54,62.5
2022-09-18,Euro,2,14161.38,2,0,14161.38,100.0
2022-09-17,UK Pound,1,11962.48,1,0,11962.48,100.0
2022-09-17,Euro,6,42451.39,6,0,42451.39,100.0
2022-09-17,Saudi Riyal,5,148582.11,5,0,148582.11,100.0
2022-09-17,US Dollar,10,88682.32,2,8,12244.98,20.0
2022-09-17,Swiss Franc,1,10286.52,1,0,10286.52,100.0
2022-09-16,Australian Dollar,1,16065.38,0,1,null,0.0
2022-09-16,US Dollar,20,217289.29,8,12,111250.22,40.0


### Validations

In [0]:
print(df.agg(sf.sum("tot_transaction_count")).first()[0])
print(df.agg(sf.sum("laundering_count")).first()[0])
print(df.agg(sf.sum("non_laundering_count")).first()[0])
print(
    df.withColumn("transaction_count_verification", (sf.col("laundering_count") + sf.col("non_laundering_count")))
    .filter(sf.col("transaction_count_verification") != sf.col("tot_transaction_count"))
    .limit(1)
    .count() > 0
)
print(
    df.groupBy("transaction_date", "payment_currency")
    .count()
    .filter(sf.col("count") > 1)
    .limit(1)
    .count() > 0
)

5078336
5177
5073159
False
False


### Ingesting data to Gold layer

In [0]:
gold_daily_transaction_metrics = "aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics"

df.write.format("delta").mode("overwrite").saveAsTable(gold_daily_transaction_metrics)

In [0]:
%sql
describe table aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics

col_name,data_type,comment
transaction_date,date,null
payment_currency,string,null
tot_transaction_count,bigint,null
total_amount_paid,"decimal(29,2)",null
laundering_count,bigint,null
non_laundering_count,bigint,null
net_laundering_amount,"decimal(29,2)",null
laundering_labelled_transaction_pct,double,null


In [0]:
%sql
select count(*) from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics

count(*)
230


In [0]:
%sql
select 
    transaction_date,
    payment_currency,
    count(*)
from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics
group by 1, 2
having count(*) > 1

transaction_date,payment_currency,count(*)


In [0]:
%sql
select sum(tot_transaction_count) from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics

sum(tot_transaction_count)
5078336


In [0]:
%sql
select sum(laundering_count) from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics

sum(laundering_count)
5177


In [0]:
%sql
select sum(non_laundering_count) from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics

sum(non_laundering_count)
5073159


In [0]:
%sql
select * from aml_transactions.aml_transactions_schema.gold_daily_transaction_metrics limit 5

transaction_date,payment_currency,tot_transaction_count,total_amount_paid,laundering_count,non_laundering_count,net_laundering_amount,laundering_labelled_transaction_pct
2022-09-18,Saudi Riyal,1,2391.92,1,0,2391.92,100.0
2022-09-18,US Dollar,8,45071.27,5,3,27839.54,62.5
2022-09-18,Euro,2,14161.38,2,0,14161.38,100.0
2022-09-17,UK Pound,1,11962.48,1,0,11962.48,100.0
2022-09-17,Euro,6,42451.39,6,0,42451.39,100.0
